# Online Course Recommendation — Group 16

**Duy Tan University, Da Nang — DS 423**  
**Trần Lãnh:** 28211151726 · **Trịnh Minh Son:** 28211144373

This presentation notebook demonstrates the verified content-based recommender implemented in `recommendation_system.py`.

## 1. Method overview

1. Combine course title, skills, provider, difficulty, and description.
2. Fit TF-IDF vocabulary and IDF weights.
3. Compute cosine similarity between a query and the catalog.
4. Rank with 95% content similarity and a 5% rating tie-breaker.
5. Return the Top-N courses.

> This is not supervised training: there is no target label, loss function, epoch, or gradient descent. `fit_transform()` learns the TF-IDF vocabulary and IDF weights.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from recommendation_system import (
    load_courses,
    build_text,
    fit_model,
    recommend_by_title,
    evaluate,
)

DATA_PATH = Path('Coursera_cleaned_for_recommendation.csv')
RANDOM_STATE = 42

## 2. Load and inspect the catalog

The cleaned dataset contains 3,424 courses and 11 columns.

In [ ]:
courses = load_courses(DATA_PATH)
print(f'Shape: {courses.shape[0]:,} courses × {courses.shape[1]} columns')
display(courses[[
    'course_id', 'course_title', 'university',
    'difficulty_level', 'course_rating_filled'
]].head())

In [ ]:
summary = pd.DataFrame({
    'Value': [
        len(courses),
        courses['course_title'].nunique(),
        courses['university'].nunique(),
        courses['course_rating'].isna().sum(),
    ]
}, index=[
    'Courses', 'Unique titles', 'Universities / organizations',
    'Missing original ratings'
])
display(summary)

## 3. Feature preparation

The title is repeated three times and skills twice to increase their influence. This is a simple, interpretable form of field weighting.

In [ ]:
combined_text = build_text(courses)
print('Example course:', courses.loc[0, 'course_title'])
print('Combined text preview:')
print(combined_text.iloc[0][:500] + ' ...')

## 4. Fit TF-IDF

Key settings: English stop words, unigrams and bigrams, `min_df=2`, `max_df=0.95`, sublinear term frequency, and at most 30,000 features.

In [ ]:
vectorizer, course_matrix = fit_model(courses)
print('TF-IDF matrix shape:', course_matrix.shape)
print('Non-zero values:', f'{course_matrix.nnz:,}')
print('Sample features:', vectorizer.get_feature_names_out()[:20])

## 5. Generate Top-N recommendations

Change `QUERY_TITLE` or `TOP_N`, then run the cell. The function removes the query itself before ranking results.

In [ ]:
QUERY_TITLE = 'AI For Everyone'
TOP_N = 5

recommendations = recommend_by_title(
    courses, course_matrix, QUERY_TITLE, n=TOP_N
)
display(recommendations[[
    'rank', 'course_title', 'university',
    'difficulty_level', 'course_rating_filled', 'score'
]])

### Try another topic

In [ ]:
recommend_by_title(
    courses, course_matrix, 'Creative Writing', n=5
)[[
    'rank', 'course_title', 'university', 'score'
]]

## 6. Offline evaluation

Because the catalog has no learner behavior, another course is treated as relevant when its skill-set Jaccard similarity to the query is at least 0.20. A fixed seed samples 250 queries; 157 have at least one relevant neighbor.

In [ ]:
metrics, protocol = evaluate(
    courses, course_matrix,
    ks=(5, 10, 20),
    sample_size=250,
    relevance_threshold=0.20,
)
display(metrics.round(3))
protocol

In [ ]:
ax = metrics.set_index('k')[[
    'precision_at_k', 'recall_at_k'
]].plot(
    kind='bar', figsize=(8, 4.5),
    color=['#1B4965', '#5FA8D3']
)
ax.set(
    title='Offline Retrieval Performance',
    xlabel='K', ylabel='Mean score', ylim=(0, 1)
)
ax.legend(['Precision@K', 'Recall@K'], frameon=False)
plt.tight_layout()
plt.show()

## 7. Interpretation and limitations

- Precision decreases as the recommendation list becomes longer.
- Recall reaches approximately 85.5% at K = 20.
- The evaluation measures consistency with skill metadata, not learner satisfaction.
- Some lexical matches can cause semantic drift.
- Future work should use real clicks, enrollments, completions, ratings, and A/B testing.

## Short answers for questions

**Where is training?** `fit_model()` calls `fit_transform()` to learn TF-IDF vocabulary and IDF weights; there is no supervised training.  
**Why not collaborative filtering?** The dataset has no learner-course interaction matrix.  
**Why cosine similarity?** It works efficiently with sparse TF-IDF vectors and is less affected by text length.  
**Why does recall increase?** A longer list retrieves more of the relevant catalog neighbors.